# Notebook 04 – NLP Analysis of Therapist Comments

This notebook applies Natural Language Processing (NLP) techniques to the free-text comments recorded by therapists during the personalised storytelling intervention. The analysis aims to identify recurring linguistic patterns, sentiment, and behavioural themes within therapist observations.

The qualitative NLP findings will subsequently be examined alongside the participant clusters identified through K-Means clustering. This enables comparison of therapist observations across the two participant trajectory groups and provides a qualitative interpretation of the behavioural patterns identified in the quantitative clustering analysis.

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [7]:
project_root = Path.cwd().parent

processed_data_folder = project_root / "data" / "processed"
outputs_folder = project_root / "outputs"
figures_folder = project_root / "figures"

In [8]:
cleaned_file = processed_data_folder / "cleaned_data.csv"

df = pd.read_csv(cleaned_file)

df.shape

(512, 75)

In [9]:
df["Submitted_by"].value_counts(dropna=False)

Submitted_by
T      256
P      240
P/C     16
Name: count, dtype: int64

## 4. Filter Therapist Records

The NLP analysis was restricted to therapist-generated observations to maintain consistency in the source and professional perspective of the qualitative data. Records submitted by parents or carers were therefore excluded from the NLP corpus.

In [10]:
therapist_df = df[df["Submitted_by"] == "T"].copy()

therapist_df.shape

(256, 75)

In [11]:
comment_cols = [
    col for col in therapist_df.columns
    if "Comment" in col
]

comment_cols

['Comment_Q1',
 'Comment_Q2',
 'Comment_Q3',
 'Comment_Q4',
 'Comment_Q5',
 'Comment_Q6',
 'Comment_Q7',
 'Comment_Q8',
 'Comment_Q9',
 'Comment_Q10',
 'Comment_Q11',
 'Comment_Q12',
 'Comment_Q13',
 'Comment_Q14',
 'Comment_Q15',
 'Comment_Q16',
 'Comment_Q17',
 'Comment_Q18',
 'Comment_Q20',
 'Comment_Q21',
 'Comment_Q22',
 'Comment_Q23',
 'Comment_Q24',
 'Comment_Q25',
 'Comment_Q26']

In [12]:
comment_counts = therapist_df[comment_cols].notna().sum()

comment_counts

Comment_Q1     24
Comment_Q2      5
Comment_Q3     36
Comment_Q4     11
Comment_Q5     16
Comment_Q6     10
Comment_Q7     15
Comment_Q8     21
Comment_Q9     36
Comment_Q10    21
Comment_Q11    19
Comment_Q12     3
Comment_Q13    17
Comment_Q14    15
Comment_Q15    28
Comment_Q16    14
Comment_Q17     3
Comment_Q18    15
Comment_Q20     8
Comment_Q21     6
Comment_Q22    12
Comment_Q23     6
Comment_Q24     9
Comment_Q25     7
Comment_Q26     2
dtype: int64

In [13]:
therapist_df["Number_of_comments"] = (
    therapist_df[comment_cols]
    .notna()
    .sum(axis=1)
)

therapist_df["Number_of_comments"].value_counts().sort_index()

Number_of_comments
0     121
1      45
2      33
3      23
4      16
5       7
6       4
7       3
8       2
9       1
10      1
Name: count, dtype: int64

In [14]:
therapist_df[comment_cols].stack().head(20)

16  Comment_Q2     The participant appeared to understand the act...
    Comment_Q8     Marked restlessness was observed, and cooperat...
    Comment_Q15                                        Didn’t answer
17  Comment_Q1     The participant entered the classroom easily. ...
    Comment_Q2     Responsiveness to the activity improved. The p...
    Comment_Q6     Due to the severity of the disorder, the parti...
    Comment_Q10    At times, the participant verbally repeated “g...
    Comment_Q13    The participant separated from the father with...
18  Comment_Q3     The participant appeared to expect repetition ...
    Comment_Q5     Voice imitation and intentional changes in ton...
    Comment_Q18    Understanding of the core theme of the story r...
    Comment_Q20               Occasionally verbalized “grandfather.”
19  Comment_Q1     Notable progress was observed in independent s...
    Comment_Q4     Verbal interaction began upon entering the cla...
    Comment_Q5             Attenti

In [16]:
therapist_df["combined_comments"] = (
    therapist_df[comment_cols]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.strip()
)

In [17]:
"combined_comments" in therapist_df.columns

True

In [18]:
therapist_df.loc[
    therapist_df["combined_comments"] != "",
    ["Participant id", "Session number", "combined_comments"]
].head()

,Participant id,Session number,combined_comments
16,102,2,The participant appeared to understand the act...
17,102,3,The participant entered the classroom easily. ...
18,102,4,The participant appeared to expect repetition ...
19,102,5,Notable progress was observed in independent s...
20,102,6,Decresed By 1 Minutes 45 sec


In [19]:
has_text = therapist_df["combined_comments"].str.strip().ne("")

print("Total therapist sessions:", len(therapist_df))
print("Sessions with comments:", has_text.sum())
print("Sessions without comments:", (~has_text).sum())

Total therapist sessions: 256
Sessions with comments: 134
Sessions without comments: 122


In [20]:
nlp_df = therapist_df.loc[
    therapist_df["combined_comments"].str.strip().ne(""),
    ["Participant id", "Session number", "combined_comments"]
].copy()

nlp_df.shape

(134, 3)

## 5. Explore the Therapist Text Corpus

Before NLP preprocessing, the availability and length of therapist-generated text were examined to assess the suitability of the qualitative data for subsequent text analysis.

In [21]:
nlp_df["word_count"] = (
    nlp_df["combined_comments"]
    .str.split()
    .str.len()
)

nlp_df["word_count"].describe()

count    134.000000
mean      33.828358
std       35.692194
min        1.000000
25%        9.000000
50%       23.000000
75%       44.250000
max      194.000000
Name: word_count, dtype: float64

In [22]:
nlp_df.loc[
    nlp_df["word_count"] <= 5,
    ["Participant id", "Session number", "word_count", "combined_comments"]
].sort_values("word_count")

,Participant id,Session number,word_count,combined_comments
100,109,6,1,Smiling
208,116,2,1,Smiling
212,116,6,1,Gestures
322,124,4,1,Smiling
386,128,4,1,Slow
256,119,2,2,1.5-2 Min
419,131,5,2,30 Seconds
180,114,6,2,Very Fast
421,131,7,2,10 seconds
177,114,3,3,Quick and complete


In [23]:
total_words = nlp_df["word_count"].sum()

print("Number of therapist text documents:", len(nlp_df))
print("Total words in corpus:", total_words)
print("Mean words per document:", round(nlp_df["word_count"].mean(), 2))
print("Median words per document:", nlp_df["word_count"].median())

Number of therapist text documents: 134
Total words in corpus: 4533
Mean words per document: 33.83
Median words per document: 23.0


## 6. Text Pre-processing

A separate cleaned version of the therapist text was created for NLP analysis while preserving the original comments. Initial preprocessing standardised letter case, removed unnecessary punctuation and non-alphabetic characters, and normalised whitespace. Further preprocessing decisions were evaluated separately to avoid removing behaviourally meaningful information from the relatively small corpus.

In [24]:
import re

In [25]:
def basic_clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [27]:
nlp_df["cleaned_text"] = nlp_df["combined_comments"].apply(basic_clean_text)

In [28]:
nlp_df[
    ["combined_comments", "cleaned_text"]
].head(10)

,combined_comments,cleaned_text
16,The participant appeared to understand the act...,the participant appeared to understand the act...
17,The participant entered the classroom easily. ...,the participant entered the classroom easily w...
18,The participant appeared to expect repetition ...,the participant appeared to expect repetition ...
19,Notable progress was observed in independent s...,notable progress was observed in independent s...
20,Decresed By 1 Minutes 45 sec,decresed by minutes sec
21,Positive affect was inferred through smiling a...,positive affect was inferred through smiling a...
22,"According to the mother, the participant greet...",according to the mother the participant greete...
23,Improvement in therapist–participant rapport w...,improvement in therapist participant rapport w...
32,The participant cooperated by memorizing and r...,the participant cooperated by memorizing and r...
33,he participant attempted to imitate the therap...,he participant attempted to imitate the therap...


In [29]:
from collections import Counter

all_words = " ".join(nlp_df["cleaned_text"]).split()

word_freq = Counter(all_words)

word_freq.most_common(30)

[('the', 416),
 ('to', 134),
 ('and', 133),
 ('participant', 123),
 ('story', 111),
 ('of', 105),
 ('he', 98),
 ('in', 94),
 ('was', 91),
 ('a', 65),
 ('this', 60),
 ('with', 49),
 ('session', 44),
 ('his', 41),
 ('s', 40),
 ('that', 36),
 ('verbal', 32),
 ('observed', 31),
 ('new', 30),
 ('at', 26),
 ('it', 26),
 ('mother', 25),
 ('were', 24),
 ('not', 24),
 ('i', 24),
 ('increased', 23),
 ('activity', 22),
 ('during', 22),
 ('is', 22),
 ('stories', 22)]

### Stopword Identification

Initial word-frequency analysis showed that the therapist comment corpus contained many high-frequency grammatical words, such as *the*, *to*, *and*, and *of*, which provide limited semantic value for identifying behavioural patterns. Standard English stopwords were therefore identified for removal during text preprocessing.

At this stage, only standard English stopwords were considered. Domain-specific terms such as *participant*, *story*, *session*, and *therapist* were retained initially so that their frequency and relevance could be evaluated before deciding whether they should also be excluded from subsequent analyses.

In [30]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stop_words = set(ENGLISH_STOP_WORDS)

len(stop_words)

318

In [31]:
def remove_stopwords(text):
    words = text.split()
    words = [
        word for word in words
        if word not in stop_words and word != "s"
    ]
    return " ".join(words)

nlp_df["text_no_stopwords"] = (
    nlp_df["cleaned_text"]
    .apply(remove_stopwords)
)

In [32]:
nlp_df[
    ["cleaned_text", "text_no_stopwords"]
].head(10)

,cleaned_text,text_no_stopwords
16,the participant appeared to understand the act...,participant appeared understand activity showe...
17,the participant entered the classroom easily w...,participant entered classroom easily limited v...
18,the participant appeared to expect repetition ...,participant appeared expect repetition previou...
19,notable progress was observed in independent s...,notable progress observed independent separati...
20,decresed by minutes sec,decresed minutes sec
21,positive affect was inferred through smiling a...,positive affect inferred smiling sustained eye...
22,according to the mother the participant greete...,according mother participant greeted grandmoth...
23,improvement in therapist participant rapport w...,improvement therapist participant rapport obse...
32,the participant cooperated by memorizing and r...,participant cooperated memorizing repeating an...
33,he participant attempted to imitate the therap...,participant attempted imitate therapist speech...


In [33]:
negation_words = ["not", "no", "never", "nor", "without"]

{word: word in stop_words for word in negation_words}

{'not': True, 'no': True, 'never': True, 'nor': True, 'without': True}

### Preserve Negation Terms

Negation terms were retained during stopword removal because they can substantially alter the meaning of behavioural observations. Words such as *not*, *no*, *never*, *nor*, and *without* were therefore excluded from the stopword list to reduce the risk of reversing the intended meaning of therapist comments.

In [34]:
negation_words = {"not", "no", "never", "nor", "without"}

stop_words = stop_words - negation_words

In [35]:
{word: word in stop_words for word in negation_words}

{'not': False, 'nor': False, 'never': False, 'no': False, 'without': False}

In [36]:
nlp_df["text_no_stopwords"] = (
    nlp_df["cleaned_text"]
    .apply(remove_stopwords)
)

In [37]:
nlp_df[
    nlp_df["cleaned_text"].str.contains(r"\b(not|no|never|nor|without)\b", regex=True)
][["cleaned_text", "text_no_stopwords"]].head(10)

C:\Users\puspi\AppData\Local\Temp\ipykernel_9976\1912149896.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  nlp_df["cleaned_text"].str.contains(r"\b(not|no|never|nor|without)\b", regex=True)


,cleaned_text,text_no_stopwords
16,the participant appeared to understand the act...,participant appeared understand activity showe...
17,the participant entered the classroom easily w...,participant entered classroom easily limited v...
18,the participant appeared to expect repetition ...,participant appeared expect repetition previou...
19,notable progress was observed in independent s...,notable progress observed independent separati...
21,positive affect was inferred through smiling a...,positive affect inferred smiling sustained eye...
32,the participant cooperated by memorizing and r...,participant cooperated memorizing repeating an...
35,this method can be recommended as an effective...,method recommended effective approach increasi...
39,this task demand was considered too advanced f...,task demand considered advanced participant cu...
67,due to family related conflicts and instabilit...,family related conflicts instability home week...
70,according to the mother s report when the part...,according mother report participant angry week...
